# 24 - Workflow Readiness and Schedule Plan

Persists and validates the production task graph:

- Hourly ingestion → Silver → incremental/full Gold → validation.
- Weekly Time Travel evidence before OPTIMIZE and 30-day VACUUM.
- Manual end-to-end evidence task for final acceptance.

The workflow plan is stored in `control.workflow_task_plan`.
Deploy the same task graph in Databricks Workflows and keep the
schedules paused until all notebooks pass interactively.


In [ ]:
%run ./05_bronze_common


In [ ]:
import uuid

from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql import types as T


dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")

catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
spark.conf.set("spark.sql.session.timeZone", "UTC")

pipeline_name = "workflow_readiness"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
target_table = f"{catalog}.control.workflow_task_plan"

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`control`.`workflow_task_plan` (
        workflow_name STRING NOT NULL,
        task_order INT NOT NULL,
        task_key STRING NOT NULL,
        notebook_path STRING NOT NULL,
        depends_on ARRAY<STRING> NOT NULL,
        schedule_type STRING NOT NULL,
        schedule_expression STRING NOT NULL,
        timezone_id STRING NOT NULL,
        parameters MAP<STRING, STRING> NOT NULL,
        enabled BOOLEAN NOT NULL,
        updated_ts TIMESTAMP NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    TBLPROPERTIES (
        'comment' = 'Deployable task dependency and schedule specification'
    )
    """
)

schema = T.StructType(
    [
        T.StructField("workflow_name", T.StringType(), False),
        T.StructField("task_order", T.IntegerType(), False),
        T.StructField("task_key", T.StringType(), False),
        T.StructField("notebook_path", T.StringType(), False),
        T.StructField("depends_on", T.ArrayType(T.StringType()), False),
        T.StructField("schedule_type", T.StringType(), False),
        T.StructField("schedule_expression", T.StringType(), False),
        T.StructField("timezone_id", T.StringType(), False),
        T.StructField("parameters", T.MapType(T.StringType(), T.StringType()), False),
        T.StructField("enabled", T.BooleanType(), False),
        T.StructField("updated_ts", T.TimestampType(), False),
        T.StructField("pipeline_run_id", T.StringType(), False),
    ]
)

hourly = "0 0 * * * ?"
weekly = "0 0 2 ? * SUN *"
timezone_id = "Asia/Ho_Chi_Minh"
common = {"catalog": catalog, "job_run_id": "{{job.run_id}}"}
now = datetime.now(timezone.utc)

task_specs = [
    ("abc_retail_hourly", 10, "bronze_clickstream", "notebooks/06_bronze_clickstream.ipynb", [], "HOURLY", hourly, common),
    ("abc_retail_hourly", 20, "bronze_orders", "notebooks/07_bronze_orders.ipynb", [], "HOURLY", hourly, common),
    ("abc_retail_hourly", 30, "bronze_customers", "notebooks/08_bronze_customers.ipynb", [], "HOURLY", hourly, common),
    ("abc_retail_hourly", 40, "silver_events", "notebooks/10_silver_events.ipynb", ["bronze_clickstream"], "HOURLY", hourly, common),
    ("abc_retail_hourly", 50, "silver_orders", "notebooks/11_silver_orders.ipynb", ["bronze_orders"], "HOURLY", hourly, common),
    ("abc_retail_hourly", 60, "silver_order_items", "notebooks/12_silver_order_items.ipynb", ["silver_orders"], "HOURLY", hourly, common),
    ("abc_retail_hourly", 70, "silver_customers", "notebooks/13_silver_customers_scd2.ipynb", ["bronze_customers"], "HOURLY", hourly, common),
    ("abc_retail_hourly", 80, "gold_funnel_cdf", "notebooks/20_incremental_funnel_cdf.ipynb", ["silver_events"], "HOURLY", hourly, common),
    ("abc_retail_hourly", 90, "gold_revenue", "notebooks/16_gold_revenue_daily.ipynb", ["silver_orders"], "HOURLY", hourly, common),
    ("abc_retail_hourly", 100, "gold_customer_360", "notebooks/17_gold_customer_360.ipynb", ["silver_events", "silver_orders", "silver_customers"], "HOURLY", hourly, common),
    ("abc_retail_hourly", 110, "gold_dq", "notebooks/18_gold_data_quality_dashboard.ipynb", ["silver_events", "silver_orders", "silver_customers"], "HOURLY", hourly, common),
    ("abc_retail_hourly", 120, "validate_gold", "notebooks/19_validate_gold.ipynb", ["gold_funnel_cdf", "gold_revenue", "gold_customer_360", "gold_dq"], "HOURLY", hourly, common),
    ("abc_retail_weekly_maintenance", 10, "time_travel_evidence", "notebooks/21_time_travel_demo.ipynb", [], "WEEKLY", weekly, common),
    ("abc_retail_weekly_maintenance", 20, "optimize_vacuum", "notebooks/22_optimize_vacuum_metrics.ipynb", ["time_travel_evidence"], "WEEKLY", weekly, {**common, "vacuum_mode": "EXECUTE"}),
    ("abc_retail_manual_evidence", 10, "end_to_end_evidence", "notebooks/23_end_to_end_tests.ipynb", [], "MANUAL", "MANUAL", common),
]

rows = [
    Row(
        workflow_name=workflow_name,
        task_order=task_order,
        task_key=task_key,
        notebook_path=notebook_path,
        depends_on=depends_on,
        schedule_type=schedule_type,
        schedule_expression=schedule_expression,
        timezone_id=timezone_id,
        parameters=parameters,
        enabled=True,
        updated_ts=now,
        pipeline_run_id=pipeline_run_id,
    )
    for (
        workflow_name, task_order, task_key, notebook_path, depends_on,
        schedule_type, schedule_expression, parameters
    ) in task_specs
]
plan = spark.createDataFrame(rows, schema)

upsert_audit(
    catalog, pipeline_run_id, pipeline_name, "task_plan", start_ts,
    "RUNNING", ["notebooks/00-24"], job_run_id,
)

try:
    task_keys = {
        row.workflow_name: {
            task.task_key
            for task in plan.filter(F.col("workflow_name") == row.workflow_name)
            .select("task_key")
            .collect()
        }
        for row in plan.select("workflow_name").distinct().collect()
    }
    invalid_dependencies = []
    for task in plan.select("workflow_name", "task_key", "depends_on").collect():
        for dependency in task.depends_on:
            if dependency not in task_keys[task.workflow_name]:
                invalid_dependencies.append(
                    f"{task.workflow_name}.{task.task_key}->{dependency}"
                )

    duplicate_tasks = (
        plan.groupBy("workflow_name", "task_key")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )
    required_hourly = {
        "bronze_clickstream", "bronze_orders", "bronze_customers",
        "silver_events", "silver_orders", "silver_order_items",
        "silver_customers", "gold_funnel_cdf", "gold_revenue",
        "gold_customer_360", "gold_dq", "validate_gold",
    }
    actual_hourly = {
        row.task_key
        for row in plan.filter(F.col("workflow_name") == "abc_retail_hourly")
        .select("task_key").collect()
    }
    missing_hourly = sorted(required_hourly - actual_hourly)
    weekly_vacuum = plan.filter(
        (F.col("workflow_name") == "abc_retail_weekly_maintenance")
        & (F.col("task_key") == "optimize_vacuum")
        & (F.col("parameters")["vacuum_mode"] == "EXECUTE")
    ).count()

    if invalid_dependencies or duplicate_tasks or missing_hourly or not weekly_vacuum:
        raise RuntimeError(
            f"Invalid workflow plan: dependencies={invalid_dependencies}, "
            f"duplicate_tasks={duplicate_tasks}, missing_hourly={missing_hourly}, "
            f"weekly_vacuum={weekly_vacuum}"
        )

    workflows = [row.workflow_name for row in plan.select("workflow_name").distinct().collect()]
    DeltaTable.forName(spark, target_table).delete(
        F.col("workflow_name").isin(workflows)
    )
    plan.write.format("delta").mode("append").saveAsTable(target_table)

    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "task_plan", start_ts,
        "SUCCESS", ["notebooks/00-24"], job_run_id,
        end_ts=datetime.now(timezone.utc), rows_read=len(task_specs),
        rows_written=len(task_specs), rows_quarantined=0,
    )
except Exception as exc:
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "task_plan", start_ts,
        "FAILED", ["notebooks/00-24"], job_run_id,
        end_ts=datetime.now(timezone.utc), error_message=str(exc)[:4000],
    )
    raise

spark.table(target_table).orderBy("workflow_name", "task_order").display()
